In [0]:
dbutils.widgets.text("table_name","")
table_name = dbutils.widgets.get("table_name")
dbutils.widgets.text("landing_timestamp","")
landing_timestamp = dbutils.widgets.get("landing_timestamp")


In [0]:
dbutils.widgets.text("load_type","")
load_type = dbutils.widgets.get("load_type")
dbutils.widgets.text("merge_keys","")
merge_keys = dbutils.widgets.get("merge_keys")


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
bronze_df=spark.read.table(f"caresyncdatabricks.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")



In [0]:
updated_df = bronze_df.withColumn("insert_timestamp", current_timestamp()).withColumn("update_timestamp", current_timestamp())

In [0]:
from delta.tables import DeltaTable

# Target silver table
target_table = f"caresyncdatabricks.silver.{table_name}"

# Make load_type case-insensitive: 'full', 'Full' -> 'FULL'
load_type = load_type.strip().upper()

# Turn "hospital_id, dept_id" into ['hospital_id', 'dept_id']
key_list = [key.strip() for key in merge_keys.split(",") if key.strip()]

print(f"Source table : caresyncdatabricks.bronze.{table_name}")
print(f"Target table : {target_table}")
print(f"Landing time : {landing_timestamp}")
print(f"Load type    : {load_type}")
print(f"Merge keys   : {key_list}")

In [0]:
print(f"Starting {load_type} load into {target_table}")

if load_type == "FULL":
    # Replace all data in the target table
    print("FULL load: overwriting all data in target table...")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("FULL load completed.")

elif load_type == "APPEND":
    # Add new rows to the target table
    print("APPEND load: adding new rows to target table...")
    updated_df.write.mode("append").saveAsTable(target_table)
    print("APPEND load completed.")

elif load_type == "MERGE":
    if not key_list:
        raise ValueError("merge_keys is required when load_type is MERGE")

    if not spark.catalog.tableExists(target_table):
        # First run: table does not exist yet, so create it
        print("Target table does not exist yet. Creating it with the current data...")
        updated_df.write.saveAsTable(target_table)
        print("Target table created.")
    else:
        # Keep one row per key (MERGE fails if a key appears twice in the source)
        source_df = updated_df.dropDuplicates(key_list)
        print(f"MERGE load: matching rows on keys {key_list}...")

        # Join condition, e.g. target.hospital_id = source.hospital_id
        join_condition = " AND ".join([f"target.{key} = source.{key}" for key in key_list])
        print(f"Join condition: {join_condition}")

        # Columns to update: everything except insert_timestamp (keep original insert time)
        update_columns = {col_name: f"source.{col_name}" for col_name in source_df.columns if col_name != "insert_timestamp"}

        target_delta = DeltaTable.forName(spark, target_table)
        (target_delta.alias("target")
            .merge(source_df.alias("source"), join_condition)
            .whenMatchedUpdate(set=update_columns)   # existing key -> update
            .whenNotMatchedInsertAll()               # new key -> insert
            .execute())

        # Show how many rows were inserted / updated by this merge
        merge_metrics = target_delta.history(1).select("operationMetrics").collect()[0][0]
        print(f"Rows inserted: {merge_metrics.get('numTargetRowsInserted')}")
        print(f"Rows updated : {merge_metrics.get('numTargetRowsUpdated')}")
        print("MERGE load completed.")

else:
    raise ValueError(f"Invalid load_type '{load_type}'. Use FULL, APPEND or MERGE.")

In [0]:
record_count=updated_df.count()
dbutils.notebook.exit(str(record_count))

In [0]:
%sql
select * from caresyncdatabricks.silver.patients